# AlphaBreast V2 - ResNet18 + Bidirectional Attention

Improvements from V1:
- ResNet18 pretrained backbone (replacing 3-layer CNN)
- Bidirectional cross-attention (CC attends to MLO AND MLO attends to CC)
- Focal Loss for class imbalance
- Mass data only (~509 pairs)
- Simple train/test split

Results: 60.3% accuracy, 0.720 AUC

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.metrics import recall_score, f1_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/CBIS-DDSM'
CSV_PATH = os.path.join(DATA_ROOT, 'csv')
JPEG_PATH = os.path.join(DATA_ROOT, 'jpeg')

OUTPUT_DIR = '/content/alphabreast_v2_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# V2: Still mass data only
mass_train = pd.read_csv(os.path.join(CSV_PATH, 'mass_case_description_train_set.csv'))
mass_test = pd.read_csv(os.path.join(CSV_PATH, 'mass_case_description_test_set.csv'))

def find_image_fixed(jpeg_path, relative_path):
    if not relative_path or pd.isna(relative_path):
        return None
    parts = str(relative_path).strip().split('/')
    for i in [1, 2]:
        if len(parts) > i:
            folder_path = os.path.join(jpeg_path, parts[i])
            if os.path.exists(folder_path):
                for f in os.listdir(folder_path):
                    if f.endswith(('.jpg', '.jpeg', '.png')):
                        return os.path.join(folder_path, f)
    return None

def create_paired_dataset(df):
    paired_samples = []
    grouped = df.groupby(['patient_id', 'left or right breast'])
    for (patient_id, laterality), group in grouped:
        cc_rows = group[group['image view'] == 'CC']
        mlo_rows = group[group['image view'] == 'MLO']
        if len(cc_rows) > 0 and len(mlo_rows) > 0:
            cc_row = cc_rows.iloc[0]
            mlo_row = mlo_rows.iloc[0]
            pathology = str(cc_row['pathology']).upper()
            if 'MALIGNANT' in pathology:
                label = 1
            elif 'BENIGN' in pathology:
                label = 0
            else:
                continue
            paired_samples.append({
                'patient_id': patient_id,
                'cc_cropped': cc_row.get('cropped image file path', ''),
                'mlo_cropped': mlo_row.get('cropped image file path', ''),
                'label': label
            })
    return pd.DataFrame(paired_samples)

paired_train = create_paired_dataset(mass_train)
paired_test = create_paired_dataset(mass_test)
print(f"Training pairs: {len(paired_train)}")
print(f"Test pairs: {len(paired_test)}")

In [ ]:
class CBISDDSMDataset(Dataset):
    def __init__(self, paired_df, jpeg_path, transform=None):
        self.jpeg_path = jpeg_path
        self.transform = transform
        self.valid_samples = []
        for idx in range(len(paired_df)):
            row = paired_df.iloc[idx]
            cc_path = find_image_fixed(jpeg_path, row.get('cc_cropped', ''))
            mlo_path = find_image_fixed(jpeg_path, row.get('mlo_cropped', ''))
            if cc_path and mlo_path:
                self.valid_samples.append({
                    'cc_path': cc_path, 'mlo_path': mlo_path, 'label': row['label']
                })
        print(f"Valid samples: {len(self.valid_samples)} / {len(paired_df)}")

    def __len__(self):
        return len(self.valid_samples)

    def __getitem__(self, idx):
        sample = self.valid_samples[idx]
        cc_img = Image.open(sample['cc_path']).convert('L')
        mlo_img = Image.open(sample['mlo_path']).convert('L')
        if self.transform:
            seed = np.random.randint(2147483647)
            torch.manual_seed(seed)
            cc_img = self.transform(cc_img)
            torch.manual_seed(seed)
            mlo_img = self.transform(mlo_img)
        return cc_img, mlo_img, torch.tensor(sample['label'], dtype=torch.long)


train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = CBISDDSMDataset(paired_train, JPEG_PATH, train_transform)
test_dataset = CBISDDSMDataset(paired_test, JPEG_PATH, test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for class imbalance."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


class ResNetEncoder(nn.Module):
    """ResNet18 encoder adapted for grayscale."""
    def __init__(self, out_features=256, pretrained=True):
        super().__init__()
        resnet = models.resnet18(pretrained=pretrained)
        # Adapt first conv for grayscale
        resnet.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.fc = nn.Linear(512, out_features)

    def forward(self, x):
        x = self.backbone(x).flatten(1)
        return self.fc(x)


class BidirectionalAttention(nn.Module):
    """V2: Bidirectional cross-attention between CC and MLO."""
    def __init__(self, embed_dim=256, num_heads=8):
        super().__init__()
        self.cc_to_mlo = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.mlo_to_cc = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm_cc = nn.LayerNorm(embed_dim)
        self.norm_mlo = nn.LayerNorm(embed_dim)

    def forward(self, cc_feat, mlo_feat):
        cc = cc_feat.unsqueeze(1)
        mlo = mlo_feat.unsqueeze(1)
        cc_attended, _ = self.cc_to_mlo(cc, mlo, mlo)
        mlo_attended, _ = self.mlo_to_cc(mlo, cc, cc)
        cc_out = self.norm_cc(cc + cc_attended).squeeze(1)
        mlo_out = self.norm_mlo(mlo + mlo_attended).squeeze(1)
        return cc_out, mlo_out


class AlphaBreastV2(nn.Module):
    """V2: ResNet18 + Bidirectional Attention."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.encoder = ResNetEncoder(256, pretrained=True)  # shared
        self.attention = BidirectionalAttention(256, 8)
        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, cc, mlo):
        cc_feat = self.encoder(cc)
        mlo_feat = self.encoder(mlo)
        cc_att, mlo_att = self.attention(cc_feat, mlo_feat)
        combined = torch.cat([cc_att, mlo_att], dim=1)
        return self.classifier(combined)


model = AlphaBreastV2().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training with Focal Loss and AdamW
criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

best_auc = 0
for epoch in range(40):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for cc, mlo, labels in train_loader:
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(cc, mlo)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    scheduler.step()

    # Evaluate
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for cc, mlo, labels in test_loader:
            cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
            outputs = model(cc, mlo)
            probs = F.softmax(outputs, dim=1)
            _, pred = outputs.max(1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds) * 100
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.5

    if auc > best_auc:
        best_auc = auc
        best_acc = acc

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/40 | Train Acc: {100.*correct/total:.1f}% | "
              f"Test Acc: {acc:.1f}%, AUC: {auc:.3f}")

print(f"\nV2 Best: Acc={best_acc:.1f}%, AUC={best_auc:.3f}")